In [14]:
# ORM_Example.ipynb
# !uv pip install ipykernel

# Django Shell을 주피터랩에서 실행할 수 있도록 환경설정
import os
import django
os.environ['DJANGO_SETTINGS_MODULE'] = "config.settings"
os.environ['DJANGO_ALLOW_ASYNC_UNSAFE'] = "true"

django.setup()

In [15]:
from polls.models import Question, Choice

# ModelClass.objects : (Model)Manager객체 -> select 처리 메소드들을 제공
type(Question.objects)

django.db.models.manager.Manager

In [16]:
# pk=1 인 질문 조회
result = Question.objects.get(pk=1)
print(type(result))

<class 'polls.models.Question'>


In [17]:
result

<Question: 1. 좋아하는 색은 무었입니까?>

In [18]:
result.pk, result.question_text, result.pub_date

(1,
 '좋아하는 색은 무었입니까?',
 datetime.datetime(2026, 7, 24, 6, 39, 38, 813245, tzinfo=datetime.timezone.utc))

In [19]:
# 조회
# - M odel Manager 메소드들을 이용해서 조회
# - `all()`: 전체 조회
# - (where 절) 조건으로 조회
#     - `filter()`: 조건이 true 행드을 조회
#     -`exclude()`: 조건이 false 행드을 조회
#     - `get()` : wh건이 true 한개 행을 조회.(조회결과가 하나일 떄 사용)
# - all, filter, exclude, get 반환타입:QuerySet
# - get 반환: 모델 객체

In [20]:
# 모든 질문들을 조회
result = Question.objects.all()
print(type(result))
print("결과개수:", len(result))
print("첫번째 조회결과:", result[0],type(result[0]) )

<class 'django.db.models.query.QuerySet'>
결과개수: 4
첫번째 조회결과: 1. 좋아하는 색은 무었입니까? <class 'polls.models.Question'>


In [21]:
for question in result:
    print(question.pk, question.question_text, question.pub_date)

1 좋아하는 색은 무었입니까? 2026-07-24 06:39:38.813245+00:00
2 싫어하는 색은 무엇입니까? 2026-07-24 06:40:27.663123+00:00
3 좋아하는 동물은 무엇입니까? 2026-07-24 06:40:44.433586+00:00
4 여행가고 싶은 나라는 어디인가요? 2026-07-24 06:40:59.045859+00:00


In [22]:
result[:2] # list

[<Question: 1. 좋아하는 색은 무었입니까?>, <Question: 2. 싫어하는 색은 무엇입니까?>]

In [23]:
result[1]
result[-1] # 음수 indexing 지원 안함


ValueError: Negative indexing is not supported.

In [24]:
result.first() # 첫번째 값
result.last() # 마지막 값

<Question: 4. 여행가고 싶은 나라는 어디인가요?>

In [27]:
Question.objects.get(pk=1) # where pk=1


<Question: 1. 좋아하는 색은 무었입니까?>

In [47]:
# Choice.objects.get(vote=0) # 조회결과가 여러개인 경우는 에러 발생
try:
    Choice.objects.get(vote=12)
except:
    print("조회 결과가 없습니다.") # 조회결과가 없는 경우 에러 발생

In [34]:
# 조회결과가 여러개인 경우
result = Choice.objects.filter(vote=0) # QuerySet
len(result)
result

<QuerySet [<Choice: 2. 검정색>, <Choice: 3. 보라색>, <Choice: 4. 빨강색>, <Choice: 5. 검정색>, <Choice: 6. 녹색>, <Choice: 7. 갈색>, <Choice: 8. 연두색>, <Choice: 9. 개>, <Choice: 10. 고양이>, <Choice: 11. 닭>, <Choice: 12. 일본>, <Choice: 13. 중국>, <Choice: 14. 러시아>, <Choice: 15. 미국>]>

In [36]:
result[0].pk, result[0].choice_text, result[0].vote

(2, '검정색', 0)

In [37]:
result[0].question

<Question: 1. 좋아하는 색은 무었입니까?>

In [38]:
result = Choice.objects.exclude(vote=0) # where not vote=0
len(result)

1

In [42]:
for c in result:
    print(c.choice_text, c.vote)

파랑색 12


In [65]:
# where 조건'
# Field명__연산자 = 비교할 값

result = Choice.objects.filter(vote=0)
result = Choice.objects.filter(vote__lt=50)
result = Choice.objects.filter(vote__lte=12)
result = Choice.objects.filter(vote__gt=12)
result = Choice.objects.filter(vote__gte=12)
# 문자열
result = Choice.objects.filter(choice_text="빨강색")
result = Choice.objects.filter(choice_text__startswith="파랑") # choice_text like "파랑%"
result = Choice.objects.filter(choice_text__endswith="색")
result = Choice.objects.filter(choice_text__contains="라")

result = Choice.objects.filter(choice_text__in=["빨강색", "보라색"])
result = Choice.objects.filter(vote__range=[10, 100])


for choice in result:
    print(choice.choice_text, choice.vote)

print(result.query) # QuerySet.query 실행된 SQL문


파랑색 12
SELECT "polls_choice"."id", "polls_choice"."choice_text", "polls_choice"."vote", "polls_choice"."question_id" FROM "polls_choice" WHERE "polls_choice"."vote" BETWEEN 10 AND 100


In [67]:
# 조건이 여러개인 경우 - AND, OR
# AND - 조건들을 나영
result = Choice.objects.filter(
    vote__lt = 100,
    choice_text__in = {"파랑색", "검정색", "개", "고양이"}
)
for choice in result:
    print(choice.choice_text, choice.vote)

print(result.query) # QuerySet.query 실행된 SQL문

파랑색 12
검정색 0
검정색 0
개 0
고양이 0
SELECT "polls_choice"."id", "polls_choice"."choice_text", "polls_choice"."vote", "polls_choice"."question_id" FROM "polls_choice" WHERE ("polls_choice"."choice_text" IN (파랑색, 검정색, 고양이, 개) AND "polls_choice"."vote" < 100)


In [ ]:
# 조건이 여러개인 경우 - AND, OR
# OR - Q 클래스에 조건을 넣어주고 "|" 연산으로 묶어준다
## Q(조건) | Q(조건)
from django.db.models import Q
# vote 가 10이하이거나 100이상
# ~Q(조건) not 조건

result = Choice.objects.filter(
    Q(vote__lte = 10) | Q(vote__gte = 100)  
    
)
for choice in result:
    print(choice.choice_text, choice.vote)

print(result.query) # QuerySet.query 실행된 SQL문

검정색 0
보라색 0
빨강색 0
검정색 0
녹색 0
갈색 0
연두색 0
개 0
고양이 0
닭 0
일본 0
중국 0
러시아 0
미국 0
SELECT "polls_choice"."id", "polls_choice"."choice_text", "polls_choice"."vote", "polls_choice"."question_id" FROM "polls_choice" WHERE ("polls_choice"."vote" <= 10 OR "polls_choice"."vote" >= 100)


In [71]:
# 결과 정렬 - order byt("기준 컬럼"): ASC 정렬, DESC: "-기준 컬럼"
result = Choice.objects.all().order_by("vote")
result = Choice.objects.all().order_by("-vote")
result = Choice.objects.all().order_by("-vote", "choice_text")

print(result.query)
for choice in result:
    print(choice.choice_text, choice.vote)



SELECT "polls_choice"."id", "polls_choice"."choice_text", "polls_choice"."vote", "polls_choice"."question_id" FROM "polls_choice" ORDER BY "polls_choice"."vote" DESC, "polls_choice"."choice_text" ASC
파랑색 12
갈색 0
개 0
검정색 0
검정색 0
고양이 0
녹색 0
닭 0
러시아 0
미국 0
보라색 0
빨강색 0
연두색 0
일본 0
중국 0


In [77]:
# 특정 컬럼들만 지정해서 조회. select 컬럼명
result = Choice.objects.all().values("pk", "choice_text")
print(type(result)) # QuerySet
print(result.query)
print(type(result[0])) # 개별결과 :dict
for value in result:
    print(value['pk'], value['choice_text'], value)

<class 'django.db.models.query.QuerySet'>
SELECT "polls_choice"."id" AS "pk", "polls_choice"."choice_text" AS "choice_text" FROM "polls_choice"
<class 'dict'>
1 파랑색 {'pk': 1, 'choice_text': '파랑색'}
2 검정색 {'pk': 2, 'choice_text': '검정색'}
3 보라색 {'pk': 3, 'choice_text': '보라색'}
4 빨강색 {'pk': 4, 'choice_text': '빨강색'}
5 검정색 {'pk': 5, 'choice_text': '검정색'}
6 녹색 {'pk': 6, 'choice_text': '녹색'}
7 갈색 {'pk': 7, 'choice_text': '갈색'}
8 연두색 {'pk': 8, 'choice_text': '연두색'}
9 개 {'pk': 9, 'choice_text': '개'}
10 고양이 {'pk': 10, 'choice_text': '고양이'}
11 닭 {'pk': 11, 'choice_text': '닭'}
12 일본 {'pk': 12, 'choice_text': '일본'}
13 중국 {'pk': 13, 'choice_text': '중국'}
14 러시아 {'pk': 14, 'choice_text': '러시아'}
15 미국 {'pk': 15, 'choice_text': '미국'}
